In [50]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [51]:
df = pd.read_csv("./data/cropdata_updated.csv")
print(f"Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head(10)

Raw Dataset Shape: 16411 rows, 7 columns



,crop ID,soil_type,Seedling Stage,MOI,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1
5,Wheat,Black Soil,Germination,6,30,65.0,1
6,Wheat,Black Soil,Germination,7,31,62.0,1
7,Wheat,Black Soil,Germination,8,32,59.0,1
8,Wheat,Black Soil,Germination,9,33,56.0,1
9,Wheat,Black Soil,Germination,10,34,53.0,1


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16411 entries, 0 to 16410
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crop ID         16411 non-null  str    
 1   soil_type       16411 non-null  str    
 2   Seedling Stage  16411 non-null  str    
 3   MOI             16411 non-null  int64  
 4   temp            16411 non-null  int64  
 5   humidity        16411 non-null  float64
 6   result          16411 non-null  int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 897.6 KB


In [53]:
df.drop_duplicates(inplace=True)
df.shape

(16283, 7)

In [54]:
df = df[df["result"] != 2]
df.shape

(15161, 7)

In [55]:
target_counts = df['result'].value_counts().sort_index()
target_pct = df['result'].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    'Count': target_counts,
    'Percentage (%)': target_pct.round(2),
    'Description': [
        '0: No Irrigation',
        '1: Standard Irrigation',
    ]
})
target_summary

,Count,Percentage (%),Description
result,,,
0,8934,58.93,0: No Irrigation
1,6227,41.07,1: Standard Irrigation


In [56]:
df.rename(columns={
    "crop ID": "crop_name",
    "soil_type": "soil_type",
    "Seedling Stage": "seedling_stage",
    "MOI": "moi",
    "temp": "temp",
    "humidity": "humidity",
    "result": "result",
}, inplace=True)
df.head()

,crop_name,soil_type,seedling_stage,moi,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1


In [57]:
GROWTH_STAGE_ORDER_MAP = {
    "Germination": 1,
    "Seedling Stage": 2,
    "Vegetative Growth / Root or Tuber Development": 3,
    "Flowering": 4,
    "Pollination": 5,
    "Fruit/Grain/Bulb Formation": 6,
    "Maturation": 7,
    "Harvest": 8,
}
df["growth_stage_order"] = (df["seedling_stage"].map(GROWTH_STAGE_ORDER_MAP).fillna(0).astype(int))

In [58]:
df["moi_temp_ratio"] = (df["moi"] / df["temp"].replace(0, np.nan)).round(4)
df["moi_humidity_index"] = ((df["moi"] * df["humidity"]) / 100.0).round(4)
df.shape

(15161, 10)

In [59]:
df.isna().sum().sum()
df

,crop_name,soil_type,seedling_stage,moi,temp,humidity,result,growth_stage_order,moi_temp_ratio,moi_humidity_index
0,Wheat,Black Soil,Germination,1,25,80.0,1,1,0.0400,0.80
1,Wheat,Black Soil,Germination,2,26,77.0,1,1,0.0769,1.54
2,Wheat,Black Soil,Germination,3,27,74.0,1,1,0.1111,2.22
3,Wheat,Black Soil,Germination,4,28,71.0,1,1,0.1429,2.84
4,Wheat,Black Soil,Germination,5,29,68.0,1,1,0.1724,3.40
...,...,...,...,...,...,...,...,...,...,...
16406,Chilli,Chalky Soil,Flowering,75,16,88.0,0,4,4.6875,66.00
16407,Chilli,Chalky Soil,Flowering,76,15,89.0,0,4,5.0667,67.64
16408,Chilli,Chalky Soil,Flowering,77,14,90.0,0,4,5.5000,69.30
16409,Chilli,Chalky Soil,Flowering,78,13,91.0,0,4,6.0000,70.98


In [60]:
numerical_cols = ["moi", "temp", "humidity", "growth_stage_order", "moi_temp_ratio", "moi_humidity_index"]
categorial_cols = ["crop_name", "soil_type", "seedling_stage"]
feature_cols = categorial_cols + numerical_cols
X = df[feature_cols]
y = df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [61]:
processor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorial_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

In [62]:
X_train_transformed = processor.fit_transform(X_train)
X_test_transformed = processor.transform(X_test)
feature_names = processor.get_feature_names_out()

X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names)